# GraphOPF code

In [1]:
# !pip install torch_geometric
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.6.0+cu121.html

# !pip install pypower
# !pip install pyrlu
# # !pip install conflictfree

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [1]:
import os
import torch

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]= "0"
# os.environ["CUDA_VISIBLE_DEVICES"] = '0, 1, 2, 3'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Device:', device)  # 출력결과: cuda
print('Count of using GPUs:', torch.cuda.device_count())   #출력결과: 1 (GPU #2 한개 사용하므로)
print('Current cuda device:', torch.cuda.current_device())  # 출력결과: 2 (GPU #2 의미)

Device: cuda
Count of using GPUs: 1
Current cuda device: 0


In [2]:
# %cd /content/drive/MyDrive/kj/GOC4601_case_real_slack
# !pwd

In [2]:
import pickle

from utils.utils5658_graphlde import ACOPFProblem
# from utils.utils4601_graphlde_test import ACOPFProblem

filepath = './data/FeasiblePairs_Case5658_20_perturb_10000_samples.mat'

data = ACOPFProblem(filename=filepath) # call ACOPFProblem class in the utils.py <== In DeepLDE code, need to modify! so messy...

save_data = False

# check the size of train/validation/test dataset.
# print("Dataset of GraphLDE: ")
# print(data.train_dataset)
# print(problem.valid_dataset)
# print(problem.test_dataset)

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
data._device = DEVICE
# Put all variables in "data" to the cuda.
for attr in dir(data):
    var = getattr(data, attr)
    if not callable(var) and not attr.startswith("__") and torch.is_tensor(var):
        try:
            setattr(data, attr, var.to(DEVICE))
        except AttributeError:
            pass


/global/u1/k/kjsong/FedOPF-APPFL/fine-tuning-task/unseen/case5658/utils/utils5658_graphlde.py:109: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.slackva = torch.tensor([np.deg2rad(ppc['bus'][self.slack, idx_bus.VA])],


In [3]:
import torch
import torch_geometric
torch.cuda.empty_cache()
import torch.optim as optim
torch.set_default_dtype(torch.float32) #  If the inputs are torch.float32, must be torch.complex64. If the inputs are torch.float64, must be torch.complex128.

from torch.utils.data import TensorDataset, DataLoader, Dataset

import numpy as np
import pickle
import time
import os
import random

from pypower.api import loadcase

from models.Edge_GNN_solver import Edge_GNNSolver

from utils.loss_fn_graphlde import total_loss, ineq_violation
from utils.log import dict_agg
from global_config import base_config, global_logger, ROOT_DIRECTORY, logging
from pathlib import Path
import pickle

# from conflictfree.grad_operator import ConFIG_update
# from conflictfree.momentum_operator import PseudoMomentumOperator
# # from conflictfree.grad_operator import ConFIGOperator
# from conflictfree.utils import get_gradient_vector,apply_gradient_vector

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Set the linear algebra solver(?)
torch._C._set_linalg_preferred_backend(torch._C._LinalgBackend.Magma) # torch._C._LinalgBackend.Magma , Cusolver
torch._C._get_linalg_preferred_backend()

# MAGMA_NOWARNING
# torch.set_warn_always(False)

[W1222 22:22:12.370431101 Context.cpp:320] Warning: torch.backends.cuda.preferred_linalg_library is an experimental feature. If you see any error or unexpected behavior when this flag is set please file an issue on GitHub. (function operator())


<_LinalgBackend.Magma: 2>

* First, let's see the information of ACOPF for targeted power network!

In [4]:
ppc = loadcase("./data/matpower/pglib_opf_case5658_epigrids.mat") # in this code, we used Pypower for loading benchmark power network.
## NOTE: the dataset we used are Pypower and PGLib, so we additionally need to check whether the targeted power network is same or not!
## e.g., IEEE 57case in Pypower has different "rate A", "rate B", "rate C" values compared to PGLib.


ng = ppc['gen'].shape[0] # number of generators.
nbus = ppc['bus'].shape[0] # number of buses.
nl = ppc['branch'].shape[0] # total number of branches and transformers.

In [5]:
from scipy.stats.qmc import LatinHypercube
train_config = {
    'probType': 'acopf',
    'useCompl': True, # boolean type: whether to use completion (DC3, DeepLDE 기술)

    # GNN model parameters
    'n_gnn_layers': 3,
    'nfeature_dim': 2, # input dim
    'efeature_dim': 4, # edge feature dim 
    'hidden_dim': 40,
    'dropout_rate': 0.1,
    'K': 10, # 6 # only for TAGConv or ChebConv and GATConv (as multi-head)

    # DeepLDE hyperparameters
    'epochs': 30, # 10 (GPU RTX 4090), 8 (GPU A100)
    'batchSize': 5, # 6, (8) (GPU RTX 4090; GPU 다운 에러 발생..), 16 (GPU A100)
    'lr': 1e-3, # 1e-3
    'lr_w': 1e-3, # 1e-3
    'weight_decay': 1e-5,

    # LDF parameters
    'rho_init': 1e-3, # 0.007 
    's_init': 0.1,
    'p_iter_max': 10,
    'warmup_iter': 10, # 20, # 0 for non-warmup start cases
    'corrEps': 1e-4, # float type: correction procedure tolerance
}

eps_converge = train_config['corrEps']
valid_eps_converge = 1e-4
nepochs = train_config['epochs']
batch_size = train_config['batchSize']

train_loss_list = []
valid_loss_list = []
valid_eval_list = []

Kshot = 20
train_len_range = (0,Kshot) ## K-shot: 20, 10, 5, 1, 0
node_means, node_stds, edge_means, edge_stds = data.input_standardization(train_len_range) # (1, 2*nbus) <= for data normalization
n_means = node_means.to(DEVICE)
n_stds = node_stds.to(DEVICE)
e_means = edge_means.to(DEVICE)
e_stds = edge_stds.to(DEVICE)

## Random sampling
# train_loader = torch_geometric.loader.DataLoader(random.sample(data.train_dataset, 100), batch_size=train_config['batchSize'], shuffle=True, drop_last=True)
## Slicing
train_loader = torch_geometric.loader.DataLoader(data.train_dataset[train_len_range[0]:train_len_range[1]], batch_size=train_config['batchSize'], shuffle=True, drop_last=True)

# ## Latin hypercube sampling
# data__ = list(range(1,101))
# n_samples = 100
# sampler = LatinHypercube(d=1)
# samples = sampler.random(n=n_samples)
# indices = np.floor(samples*len(data__)).astype(int).flatten()
# trained_sample_data = [data.train_dataset[i] for i in indices]
# train_loader = torch_geometric.loader.DataLoader(trained_sample_data, batch_size=train_config['batchSize'], shuffle=True, drop_last=True)

valid_loader = torch_geometric.loader.DataLoader(data.valid_dataset, batch_size=train_config['batchSize'], shuffle=False, drop_last=True)

# solver_net = GNNSolver(data, train_config)
solver_net = Edge_GNNSolver(data, train_config)

solver_net.to(DEVICE)

print(solver_net)
num_params = sum(p.numel() for p in solver_net.parameters() if p.requires_grad)
print('The number of parameters of model is', num_params)

Edge_GNNSolver(
  (layers): ModuleList(
    (0): EdgeAggregation()
    (1): TransformerConv(120, 40, heads=10)
    (2): EdgeAggregation()
    (3): TransformerConv(120, 40, heads=10)
    (4): EdgeAggregation()
    (5): TransformerConv(120, 40, heads=10)
  )
  (flatten): Linear(in_features=12600, out_features=209, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)
The number of parameters of model is 3095929


In [6]:
#################################### SETTING THE LOGS ####################################
result_path = os.path.join(ROOT_DIRECTORY, "results")
folder_name = "FT_unseen_data_" + str(Kshot) + "_shot_" + str(train_config["batchSize"]) + "_bs_" + str(train_config["epochs"]) + "_epochs_" + str(train_config["lr"]) + "_lr_" \
              + str(train_config["rho_init"]) + "_rho_init_" + str(train_config["warmup_iter"]) + "_warmup_iter" 
# folder_name = "Local_unseen_data_" + str(Kshot) + "_shot_" + str(train_config["batchSize"]) + "_bs_" + str(train_config["epochs"]) + "_epochs_" + str(train_config["lr"]) + "_lr_" \
#                + str(train_config["rho_init"]) + "_rho_init_" + str(train_config["warmup_iter"]) + "_warmup_iter"

log_path = os.path.join(result_path, folder_name, "logs")
data_tracking_path = os.path.join(result_path, folder_name, "data_tracking")

Path(log_path).mkdir(parents=True, exist_ok=True)
Path(data_tracking_path).mkdir(parents=True, exist_ok=True)
fileh = logging.FileHandler(os.path.join(log_path, "log.txt"), 'a')
global_logger.addHandler(fileh)

* Load pretrained GraphOPF model

In [7]:
pretrained_global_graphlde = torch.load("./models/global_model/checkpoint_Global.pth", weights_only=False)

* Transfer the weights from the pretrained GraphOPF

In [8]:
global_pretrained_state_dict = pretrained_global_graphlde
target_state_dict = solver_net.state_dict()

layer_names = list(target_state_dict.keys())
# load global model
for name in layer_names[:-2]:
    if name in target_state_dict.keys() and name in global_pretrained_state_dict.keys():
        # print(name)
        target_state_dict[name] = global_pretrained_state_dict[name].clone()

solver_net.load_state_dict(target_state_dict)

# # Freezing the half-GNN layers
# for i, (name, param) in enumerate(solver_net.named_parameters()):
#     if i <= 32: # 5, 10, 21
#         # print(name)
#         param.requires_grad = False
#     else:
#         print(name)
#         param.requires_grad = True

<All keys matched successfully>

* Training method: LD framework

In [9]:
stats = {}

# NOTE: LDF parameters.
LagM_sp_gen = torch.ones(1, 2).to(DEVICE) # shape: (1, num_inequalities)
LagM_gen = torch.ones(1, 2*ng).to(DEVICE) # shape: (1, num_inequalities)
LagM_bus = torch.ones(1, 2*nbus).to(DEVICE) # shape: (1, num_inequalities)
LagM_line = torch.ones(1, 2*nl).to(DEVICE) # shape: (1, num_inequalities)

warmup_iter = train_config["warmup_iter"] # the warmup period: the NN is trained with an additional inner iteration before the first outer iteration.
rho_init = train_config["rho_init"]
s_init = train_config["s_init"]

rho = rho_init
s = s_init

rho_iter = 0
s_iter = 0

p_iter_max = train_config["p_iter_max"]
p_iter_max_sum = p_iter_max

d = 0 # 0 for static case otherwise use 5
beta = 0 # 0.001 

lr_w = train_config["lr_w"] # 이거 증가해도 되지 않을지?
lr = train_config["lr"]

print_interval = 1
epoch_stats = {}
train_start_time = time.time()
for i in range(nepochs):
    ################### TRAINING PHASE ###################
    solver_net.train()
    if i<warmup_iter:
        ######### WARM-UP PERIOD #########
        if i == 0:
            solver_opt = optim.Adam(solver_net.parameters(), lr=lr_w, weight_decay=train_config["weight_decay"]) # this will be reinitalized after warmup stage
            print("Warmup start!")

        for Xtrain in train_loader:
            Xtrain = Xtrain.to(DEVICE)
            # start_time = time.time()
            solver_opt.zero_grad()

            # DNN + NR (equality constraint)
            Yhat_train = solver_net(Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
            # Yhat_train = solver_net(Xtrain, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
            # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)

            # train_loss, train_obj, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM) # LagM is lagrangian multiplier, and the shape is (1, num_inequalities)
            train_loss, train_obj, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM_sp_gen, LagM_gen, LagM_bus, LagM_line)

            train_loss.sum().backward()
            solver_opt.step()

            dict_agg(epoch_stats, 'train_loss', train_loss.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_obj', train_obj.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_max', torch.max(ineq_dist, dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_mean', torch.mean(ineq_dist, dim=1).detach().cpu().numpy())

            ineq_p_g = ineq_dist[:,:2]
            ineq_q_g = ineq_dist[:,2:2+2*ng]
            ineq_v_m = ineq_dist[:,2+2*ng:2+2*ng+2*nbus]
            ineq_line_l = ineq_dist[:,2+2*ng+2*nbus:]

            dict_agg(epoch_stats, 'train_ineq_p_g_num_viol_0', torch.sum(ineq_p_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_q_g_num_viol_0', torch.sum(ineq_q_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_v_m_num_viol_0', torch.sum(ineq_v_m > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_line_thermal_num_viol_0', torch.sum(ineq_line_l > eps_converge, dim=1).detach().cpu().numpy())

            dict_agg(epoch_stats, 'train_eq_max', torch.max(torch.abs(eq_resid), dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_eq_mean', torch.mean(torch.abs(eq_resid), dim=1).detach().cpu().numpy())

    else:
        if i == warmup_iter:
            print("Warmup ended!")
            # data.ref_freedom = False

            # ineq_lag_flag = True # for the first warmup end epoch, consider lag. multipliers update of ineq.
        # elif (i - warmup_iter)%ineq_lag_flag_trigger == 0:
        #     # print("Doing test... continue this case..")
        #     print("consider lagrangian multipliers update for ineq. constraints!")
        #     ineq_lag_flag = True
        # else:
        #     ineq_lag_flag = False

        # for every (updated) p_iter_max_sum time.
        if (i - warmup_iter)%p_iter_max_sum == 0:
            ######### Outer Interation: calculate step size of lagrangian multipliers update #########
            if i> warmup_iter:
                print("current epoch %d || p_iter_max updated : %d -> %d" %(i, p_iter_max, p_iter_max + d))
                # s = s_init * (1/(1+beta*(s_iter + 1))) # 근데 이 부분 중복아닌가?? 있어야 하나????
                # s_iter += 1
                # print("mu iter updated : %d -> %d" %(s_iter-1, s_iter))

                rho = rho_init * (1/(1+beta*(rho_iter + 1))) # 근데 이 부분 중복아닌가?? 있어야 하나????
                rho_iter += 1
                print("rho iter updated : %d -> %d" %(rho_iter-1, rho_iter))
                p_iter_max = p_iter_max + d
                p_iter_max_sum += p_iter_max

            with torch.no_grad():
                print("Lambda updated at %d epoch" %i)
                solver_net.eval()
                for Xtrain in train_loader:
                    Xtrain = Xtrain.to(DEVICE)
                    #solver_opt.zero_grad()
                    Yhat_train = solver_net(Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
                    # Yhat_train = solver_net(Xtrain, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
                    # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)

                    LagM_sp_gen += rho*ineq_violation(data, Xtrain.x, Yhat_train)[:2] # Update the lagrangian multiplier.
                    LagM_gen += rho*ineq_violation(data, Xtrain.x, Yhat_train)[2:2+2*ng] # Update the lagrangian multiplier.
                    LagM_bus += rho*ineq_violation(data, Xtrain.x, Yhat_train)[2+2*ng:2+2*ng+2*nbus] # Update the lagrangian multiplier.
                    LagM_line += rho*ineq_violation(data, Xtrain.x, Yhat_train)[2+2*ng+2*nbus:] # Update the lagrangian multiplier.

            # What about removing this line?
            solver_opt = optim.Adam(solver_net.parameters(), lr = lr, weight_decay=train_config["weight_decay"])

        solver_net.train()
        for Xtrain in train_loader:
            Xtrain = Xtrain.to(DEVICE)
            solver_opt.zero_grad()
            Yhat_train = solver_net(Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
            # Yhat_train = solver_net(Xtrain, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
            # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)

            # train_loss, obj_train, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM) # LagM is lagrangian multiplier (1, num_inequalities)
            train_loss, train_obj, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM_sp_gen, LagM_gen, LagM_bus, LagM_line)

            train_loss.sum().backward()
            solver_opt.step()

            dict_agg(epoch_stats, 'train_loss', train_loss.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_obj', train_obj.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_max', torch.max(ineq_dist, dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_mean', torch.mean(ineq_dist, dim=1).detach().cpu().numpy())

            ineq_p_g = ineq_dist[:,:2]
            ineq_q_g = ineq_dist[:,2:2+2*ng]
            ineq_v_m = ineq_dist[:,2+2*ng:2+2*ng+2*nbus]
            ineq_line_l = ineq_dist[:,2+2*ng+2*nbus:]

            dict_agg(epoch_stats, 'train_ineq_p_g_num_viol_0', torch.sum(ineq_p_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_q_g_num_viol_0', torch.sum(ineq_q_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_v_m_num_viol_0', torch.sum(ineq_v_m > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_line_thermal_num_viol_0', torch.sum(ineq_line_l > eps_converge, dim=1).detach().cpu().numpy())

            dict_agg(epoch_stats, 'train_eq_max', torch.max(torch.abs(eq_resid), dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_eq_mean', torch.mean(torch.abs(eq_resid), dim=1).detach().cpu().numpy())

    if (i == 0) or (i%print_interval == 0):
        print(
            'Epoch {}: train loss {:.4f}, train obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq p_g num viol {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, ineq line_theraml num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}'.format(
                i, np.mean(epoch_stats['train_loss']), np.mean(epoch_stats['train_obj']), np.mean(epoch_stats['train_ineq_max']), np.mean(epoch_stats['train_ineq_mean']),
                np.mean(epoch_stats['train_ineq_p_g_num_viol_0']), np.mean(epoch_stats['train_ineq_q_g_num_viol_0']), np.mean(epoch_stats['train_ineq_v_m_num_viol_0']), np.mean(epoch_stats['train_ineq_line_thermal_num_viol_0']),
                np.mean(epoch_stats['train_eq_max']), np.mean(epoch_stats['train_eq_mean'])))

    global_logger.info('Epoch {}: train loss {:.4f}, train obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq p_g num viol {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, ineq line_theraml num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}'.format(
                i, np.mean(epoch_stats['train_loss']), np.mean(epoch_stats['train_obj']), np.mean(epoch_stats['train_ineq_max']), np.mean(epoch_stats['train_ineq_mean']),
                np.mean(epoch_stats['train_ineq_p_g_num_viol_0']), np.mean(epoch_stats['train_ineq_q_g_num_viol_0']), np.mean(epoch_stats['train_ineq_v_m_num_viol_0']), np.mean(epoch_stats['train_ineq_line_thermal_num_viol_0']),
                np.mean(epoch_stats['train_eq_max']), np.mean(epoch_stats['train_eq_mean'])))

    train_loss_list.append(np.mean(epoch_stats['train_loss']))
    # valid_eval_list.append(np.mean(epoch_stats['valid_eval']))
train_end_time = time.time()
total_train_time = train_end_time - train_start_time

Warmup start!
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It m

Epoch 0: train loss 36951.2109, train obj 1280.7169, ineq max 15744.5020, ineq mean 1.2022, ineq p_g num viol 1.0000, ineq q_g num viol 41.8000, ineq v_m num viol 0.0000, ineq line_theraml num viol 19.9500, eq max 0.0041, eq mean 0.0000


Epoch 0: train loss 36951.2109, train obj 1280.7169, ineq max 15744.5020, ineq mean 1.2022, ineq p_g num viol 1.0000, ineq q_g num viol 41.8000, ineq v_m num viol 0.0000, ineq line_theraml num viol 19.9500, eq max 0.0041, eq mean 0.0000
=
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good perf

Epoch 1: train loss 33513.5469, train obj 1237.8770, ineq max 14139.0674, ineq mean 1.0877, ineq p_g num viol 1.0000, ineq q_g num viol 36.6000, ineq v_m num viol 0.0000, ineq line_theraml num viol 16.9750, eq max 0.0038, eq mean 0.0000


Epoch 1: train loss 33513.5469, train obj 1237.8770, ineq max 14139.0674, ineq mean 1.0877, ineq p_g num viol 1.0000, ineq q_g num viol 36.6000, ineq v_m num viol 0.0000, ineq line_theraml num viol 16.9750, eq max 0.0038, eq mean 0.0000
e
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to

...

Epoch 27: train loss 6185.9600, train obj 1158.8436, ineq max 1634.6899, ineq mean 0.1260, ineq p_g num viol 1.0000, ineq q_g num viol 26.9946, ineq v_m num viol 0.0000, ineq line_theraml num viol 5.4911, eq max 0.0040, eq mean 0.0000
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performa

Epoch 28: train loss 6017.7549, train obj 1161.6658, ineq max 1579.1083, ineq mean 0.1217, ineq p_g num viol 1.0000, ineq q_g num viol 26.7690, ineq v_m num viol 0.0000, ineq line_theraml num viol 5.3121, eq max 0.0040, eq mean 0.0000


Epoch 28: train loss 6017.7549, train obj 1161.6658, ineq max 1579.1083, ineq mean 0.1217, ineq p_g num viol 1.0000, ineq q_g num viol 26.7690, ineq v_m num viol 0.0000, ineq line_theraml num viol 5.3121, eq max 0.0040, eq mean 0.0000
 WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performanc

Epoch 29: train loss 5860.2520, train obj 1164.0640, ineq max 1527.1693, ineq mean 0.1177, ineq p_g num viol 1.0000, ineq q_g num viol 26.4383, ineq v_m num viol 0.0000, ineq line_theraml num viol 5.1450, eq max 0.0040, eq mean 0.0000


Epoch 29: train loss 5860.2520, train obj 1164.0640, ineq max 1527.1693, ineq mean 0.1177, ineq p_g num viol 1.0000, ineq q_g num viol 26.4383, ineq v_m num viol 0.0000, ineq line_theraml num viol 5.1450, eq max 0.0040, eq mean 0.0000


In [10]:
total_train_time

461.5150856971741

In [11]:
# Save the training history
# train_loss_list
# epoch_stats['train_loss']
(np.array(train_loss_list)).tolist()
global_logger.info("train_loss_list:{}".format((np.array(train_loss_list)).tolist()))

data_tracking = {
                "train_loss_list": (np.array(train_loss_list)).tolist(),
                }
with open(os.path.join(data_tracking_path, "metrics.pickle"), 'wb') as handle:
    pickle.dump(data_tracking, handle, protocol=pickle.HIGHEST_PROTOCOL)

train_loss_list:[36951.2109375, 33513.546875, 26561.974609375, 21549.85546875, 18389.064453125, 15923.837890625, 13939.185546875, 12515.6162109375, 11258.1279296875, 10295.853515625, 11167.4287109375, 11587.794921875, 11282.4921875, 10725.1376953125, 10196.4736328125, 9689.7158203125, 9208.17578125, 8769.9990234375, 8375.822265625, 8021.5400390625, 7761.68408203125, 7497.61669921875, 7237.3779296875, 6992.2109375, 6768.53955078125, 6559.78271484375, 6366.2734375, 6185.9599609375, 6017.7548828125, 5860.251953125]


* Evaluation (using validation set)

In [12]:
len(data.test_dataset)

200

In [27]:
# load pretrained_model
# solver_net = torch.load(r'./models/pretrained_models/graphlde_4601_pretrained_model_20_real_slack_chebconv.pt', weights_only=False)
# solver_net = torch.load(r'./models/pretrained_models/graphlde_4601_pretrained_model_20_real_slack_chebconv_(weight_init_default).pt', weights_only=False)


In [13]:
from pypower.api import makeYbus
Ybus, Yf, Yt = makeYbus(data.baseMVA, data.ppc['bus'], data.ppc['branch'])
# branch thermal limit information
flow_max = (data.ppc['branch'][:, 5] / data.baseMVA)**2
flow_max[flow_max == 0] = np.inf # np.Inf
flow_max = torch.tensor(flow_max, dtype=torch.float32).to(data.device)

test_len = 0
node_means, node_stds, edge_means, edge_stds = data.input_standardization(test_len, train=False) # (1, 2*nbus) <= for data normalization
n_means = node_means.to(DEVICE)
n_stds = node_stds.to(DEVICE)
e_means = edge_means.to(DEVICE)
e_stds = edge_stds.to(DEVICE)

test_loader = torch_geometric.loader.DataLoader(data.test_dataset[test_len:], batch_size=1, shuffle=False, drop_last=True)
# test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

solver_net.eval()
test_stats = {}
test_eps_converge = 1e-4

LagM_sp_g = torch.ones(1, 2).to(DEVICE) # shape: (1, num_inequalities)
LagM_q_g = torch.ones(1, 2*ng).to(DEVICE) # shape: (1, num_inequalities)
LagM_v_m = torch.ones(1, 2*nbus).to(DEVICE) # shape: (1, num_inequalities)
LagM_line_l = torch.ones(1, 2*nl).to(DEVICE) # shape: (1, num_inequalities)

solve_time = []
for (i, Xtest) in enumerate(test_loader):
    Xtest = Xtest.to(DEVICE)

    start_time = time.time()
    Y = solver_net(Xtest, n_means, n_stds, e_means, e_stds)
    # Y = solver_net(Xtest, e_means, e_stds)
    end_time = time.time()

    solve_time += [end_time - start_time]

    ## line thermal limit
    pg, qg, vm, va = data.get_yvars(Y)
    vr = vm*torch.cos(va)
    vi = vm*torch.sin(va)
    vz = torch.complex(vr, vi) # complex voltage

    # calculate the branch current of from bus and to bus based on the Yf*V and Yt*V
    If = torch.tensor(Yf.todense(), dtype=torch.complex64).to(data.device) @ vz.T
    It = torch.tensor(Yt.todense(), dtype=torch.complex64).to(data.device) @ vz.T

    # Calculate the apparent power S
    Sf = vz[:,data.ppc['branch'][:,0].astype(int)] * torch.conj(If.T)
    St = vz[:,data.ppc['branch'][:,1].astype(int)] * torch.conj(It.T)
    Sff = Sf * torch.conj(Sf)
    Stt = St * torch.conj(St)

    # calculate the line thermal limit constraints violation
    diff_Sf = Sff.real - flow_max
    diff_St = Stt.real - flow_max
    # diff_Sf[torch.clamp(diff_Sf, 0) != 0]

    line_limit_vio_Sf = torch.clamp(diff_Sf, 0)
    line_limit_vio_St = torch.clamp(diff_St, 0)
    ###########################################

    test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = total_loss(data, Xtest.x, Y, LagM_sp_g, LagM_q_g, LagM_v_m, LagM_line_l) # LagM is lagrangian multiplier, and the shape is (1, num_inequalities)

    dict_agg(test_stats, 'time', end_time - start_time, op='sum')

    test_ineq_p_g = torch.cat([pg - data.pmax, data.pmin - pg], dim=1)
    test_ineq_p_g = torch.clamp(test_ineq_p_g, 0).to(data.device)
    test_ineq_q_g = test_ineq_dist[:,2:2+2*ng]
    test_ineq_v_m = test_ineq_dist[:,2+2*ng:2+2*ng+2*nbus]
    test_ineq_line_l = test_ineq_dist[:,2+2*ng+2*nbus:]

    dict_agg(test_stats, 'test_loss', test_loss.detach().cpu().numpy())
    # dict_agg(test_stats, 'test_loss', (test_loss[0]+test_loss[1]+test_loss[2]+test_loss[3]).detach().cpu().numpy())

    dict_agg(test_stats, 'test_obj_cost', test_obj_cost.detach().cpu().numpy())

    dict_agg(test_stats, 'test_ineq_max', torch.max(test_ineq_dist, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_mean', torch.mean(test_ineq_dist, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_p_g_max', torch.max(test_ineq_p_g, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_p_g_mean', torch.mean(test_ineq_p_g, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_q_g_max', torch.max(test_ineq_q_g, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_q_g_mean', torch.mean(test_ineq_q_g, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_v_m_max', torch.max(test_ineq_v_m, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_v_m_mean', torch.mean(test_ineq_v_m, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_line_l_max', torch.max(test_ineq_line_l, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_line_l_mean', torch.mean(test_ineq_line_l, dim=1).detach().cpu().numpy())

    pg_rate_torch = (((pg <= data.pmax) & (pg >= data.pmin)).sum()/ng)*100
    qg_rate_torch = (((qg <= data.qmax) & (qg >= data.qmin)).sum()/ng)*100
    dict_agg(test_stats, 'test_p_g_satisfication rate (%)', pg_rate_torch.detach().cpu().numpy().reshape(-1,1))
    dict_agg(test_stats, 'test_q_g_satisfication rate (%)', qg_rate_torch.detach().cpu().numpy().reshape(-1,1))
    # dict_agg(test_stats, 'test_p_g_satisfication rate (%)', ((torch.sum(test_ineq_p_g == 0, dim=1)/test_ineq_p_g.shape[1])*100).detach().cpu().numpy())
    # dict_agg(test_stats, 'test_q_g_satisfication rate (%)', ((torch.sum(test_ineq_q_g == 0, dim=1)/test_ineq_q_g.shape[1])*100).detach().cpu().numpy())

    v_rate_torch = (((vm <= data.vmax) & (vm >= data.vmin)).sum()/nbus)*100
    dict_agg(test_stats, 'test_v_m_satisfication rate (%)', v_rate_torch.detach().cpu().numpy().reshape(-1,1))
    # dict_agg(test_stats, 'test_v_m_satisfication rate (%)', ((torch.sum(test_ineq_v_m == 0, dim=1)/test_ineq_v_m.shape[1])*100).detach().cpu().numpy())

    sff_rate_torch = ((Sff.real <= flow_max).sum()/nl)*100        
    stt_rate_torch = ((Stt.real <= flow_max).sum()/nl)*100        
    dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', sff_rate_torch.detach().cpu().numpy().reshape(-1,1))
    dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', stt_rate_torch.detach().cpu().numpy().reshape(-1,1))
    dict_agg(test_stats, 'test_line_limit_satisfication_rate (%)', ((torch.sum(test_ineq_line_l == 0, dim=1)/test_ineq_line_l.shape[1])*100).detach().cpu().numpy())
    # dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', ((torch.sum(line_limit_vio_Sf == 0, dim=1)/line_limit_vio_Sf.shape[1])*100).detach().cpu().numpy())
    # dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', ((torch.sum(line_limit_vio_St == 0, dim=1)/line_limit_vio_St.shape[1])*100).detach().cpu().numpy())

    dict_agg(test_stats, 'test_ineq_q_g_num_viol_0', torch.sum(test_ineq_q_g > test_eps_converge, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_v_m_num_viol_0', torch.sum(test_ineq_v_m > test_eps_converge, dim=1).detach().cpu().numpy())

    test_eq_real = test_eq_resid[:,:nbus]
    test_eq_react = test_eq_resid[:,nbus:]
    dict_agg(test_stats, 'test_eq_max', torch.max(torch.abs(test_eq_resid), dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_mean', torch.mean(torch.abs(test_eq_resid), dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_real_max', torch.max(torch.abs(test_eq_real), dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_real_mean', torch.mean(torch.abs(test_eq_real), dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_react_max', torch.max(torch.abs(test_eq_react), dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_react_mean', torch.mean(torch.abs(test_eq_react), dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_active_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,:nbus] <= 1e-2) & (test_eq_resid[:,:nbus] >= -1e-2)  , dim=1)/test_eq_resid[:,:nbus].shape[1]*100).detach().cpu().numpy())
    dict_agg(test_stats, 'test_reactive_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,nbus:] <= 1e-2) & (test_eq_resid[:,nbus:] >= -1e-2)  , dim=1)/test_eq_resid[:,nbus:].shape[1]*100).detach().cpu().numpy())

    print('Test batch {}: test loss {:.4f}, test obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}, p_g satisfication rate {:.4f}, q_g satisfication rate {:.4f}, v_m satisfication rate {:.4f}, test_line_limit_satisfication_rate {:.4f}, test_line_limit_satisfication_rate_Sf {:.4f}, test_line_limit_satisfication_rate_St {:.4f}, active eq satisfication rate {:.4f}, reactive eq satisfication rate {:.4f}'.format(
                i, np.mean(test_stats['test_loss']), np.mean(test_stats['test_obj_cost']), np.mean(test_stats['test_ineq_max']), np.mean(test_stats['test_ineq_mean']),
                np.mean(test_stats['test_ineq_q_g_num_viol_0']), np.mean(test_stats['test_ineq_v_m_num_viol_0']),
                np.mean(test_stats['test_eq_max']), np.mean(test_stats['test_eq_mean']), np.mean(test_stats['test_p_g_satisfication rate (%)']), np.mean(test_stats['test_q_g_satisfication rate (%)']), np.mean(test_stats['test_v_m_satisfication rate (%)']), np.mean(test_stats['test_line_limit_satisfication_rate (%)']), np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']), np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']), np.mean(test_stats['test_active_eq_satisfication rate (%)']), np.mean(test_stats['test_reactive_eq_satisfication rate (%)'])))


Test batch 0: test loss 1248.9246, test obj 1224.6042, ineq max 20.6982, ineq mean 0.0008, ineq q_g num viol 18.0000, ineq v_m num viol 0.0000, eq max 0.0063, eq mean 0.0000, p_g satisfication rate 99.0476, q_g satisfication rate 82.8571, v_m satisfication rate 100.0000, test_line_limit_satisfication_rate 100.0000, test_line_limit_satisfication_rate_Sf 100.0000, test_line_limit_satisfication_rate_St 100.0000, active eq satisfication rate 100.0000, reactive eq satisfication rate 100.0000
Test batch 1: test loss 1250.1013, test obj 1226.2599, ineq max 20.2376, ineq mean 0.0008, ineq q_g num viol 18.5000, ineq v_m num viol 0.0000, eq max 0.0056, eq mean 0.0000, p_g satisfication rate 99.0476, q_g satisfication rate 82.3810, v_m satisfication rate 100.0000, test_line_limit_satisfication_rate 100.0000, test_line_limit_satisfication_rate_Sf 100.0000, test_line_limit_satisfication_rate_St 100.0000, active eq satisfication rate 100.0000, reactive eq satisfication rate 100.0000
Test batch 2: te

* Arithmetic mean

In [14]:
## Calculate the results of GraphLDE

print("GraphLDE obj. value for test samples: ", round(np.mean(test_stats['test_obj_cost'])*10000, 4))
print("GraphLDE eq. mean for test samples: ", np.mean(test_stats['test_eq_mean']))
print("GraphLDE eq. max for test samples: ", np.mean(test_stats['test_eq_max']))
print("GraphLDE eq. active mean for test samples: ", np.mean(test_stats['test_eq_real_mean']))
print("GraphLDE eq. active max for test samples: ", np.mean(test_stats['test_eq_real_max']))
print("GraphLDE eq. reactive mean for test samples: ", np.mean(test_stats['test_eq_react_mean']))
print("GraphLDE eq. reactive max for test samples: ", np.mean(test_stats['test_eq_react_max']))

print("\n")
print("GraphLDE ineq. mean for test samples: ", np.mean(test_stats['test_ineq_mean']))
print("GraphLDE ineq. max for test samples: ", np.mean(test_stats['test_ineq_max']))
print("GraphLDE ineq. p_g mean for test samples: ", np.mean(test_stats['test_ineq_p_g_mean']))
print("GraphLDE ineq. p_g max for test samples: ", np.mean(test_stats['test_ineq_p_g_max']))
print("GraphLDE ineq. q_g mean for test samples: ", np.mean(test_stats['test_ineq_q_g_mean']))
print("GraphLDE ineq. q_g max for test samples: ", np.mean(test_stats['test_ineq_q_g_max']))
print("GraphLDE ineq. v_m mean for test samples: ", np.mean(test_stats['test_ineq_v_m_mean']))
print("GraphLDE ineq. v_m max for test samples: ", np.mean(test_stats['test_ineq_v_m_max']))
print("GraphLDE ineq. line_l mean for test samples: ", np.mean(test_stats['test_ineq_line_l_mean']))
print("GraphLDE ineq. line_l max for test samples: ", np.mean(test_stats['test_ineq_line_l_max']))

print("\n")
print("GraphLDE p_g satisfication rate for test samples: ", np.mean(test_stats['test_p_g_satisfication rate (%)']))
print("GraphLDE q_g satisfication rate for test samples: ", np.mean(test_stats['test_q_g_satisfication rate (%)']))
print("GraphLDE v_m satisfication rate for test samples: ", np.mean(test_stats['test_v_m_satisfication rate (%)']))
print("GraphLDE test_line_limit_satisfication_rate_Sf for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']))
print("GraphLDE test_line_limit_satisfication_rate_St for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']))
print("GraphLDE active eq satisfication rate for test samples: ", np.mean(test_stats['test_active_eq_satisfication rate (%)']))
print("GraphLDE reactive eq satisfication rate for test samples: ", np.mean(test_stats['test_reactive_eq_satisfication rate (%)']))

print("\n")
# print("DeepLDE time (ms) <== average value for test dataset:", (test_stats['time']/1000)*1e3)
print("GraphLDE time (ms) <== average value for test dataset:", (np.mean(solve_time)/1)*1e3)

global_logger.info('GraphLDE obj. value for test samples: {}'.format(round(np.mean(test_stats['test_obj_cost'])*10000, 4)))
global_logger.info('GraphLDE eq. mean for test samples: {}'.format(np.mean(test_stats['test_eq_mean'])))
global_logger.info('GraphLDE eq. max for test samples: {}'.format(np.mean(test_stats['test_eq_max'])))
global_logger.info('GraphLDE eq. active mean for test samples: {}'.format(np.mean(test_stats['test_eq_real_mean'])))
global_logger.info('GraphLDE eq. active max for test samples: {}'.format(np.mean(test_stats['test_eq_real_max'])))
global_logger.info('GraphLDE eq. reactive mean for test samples: {}'.format(np.mean(test_stats['test_eq_react_mean'])))
global_logger.info('GraphLDE eq. reactive max for test samples: {}'.format(np.mean(test_stats['test_eq_react_max'])))
global_logger.info('\nGraphLDE ineq. mean for test samples: {}'.format(np.mean(test_stats['test_ineq_mean'])))
global_logger.info('GraphLDE ineq. max for test samples: {}'.format(np.mean(test_stats['test_ineq_max'])))
global_logger.info('GraphLDE ineq. p_g mean for test samples: {}'.format(np.mean(test_stats['test_ineq_p_g_mean'])))
global_logger.info('GraphLDE ineq. p_g max for test samples: {}'.format(np.mean(test_stats['test_ineq_p_g_max'])))
global_logger.info('GraphLDE ineq. q_g mean for test samples: {}'.format(np.mean(test_stats['test_ineq_q_g_mean'])))
global_logger.info('GraphLDE ineq. q_g max for test samples: {}'.format(np.mean(test_stats['test_ineq_q_g_max'])))
global_logger.info('GraphLDE ineq. v_m mean for test samples: {}'.format(np.mean(test_stats['test_ineq_v_m_mean'])))
global_logger.info('GraphLDE ineq. v_m max for test samples: {}'.format(np.mean(test_stats['test_ineq_v_m_max'])))
global_logger.info('GraphLDE ineq. line_l mean for test samples: {}'.format(np.mean(test_stats['test_ineq_line_l_mean'])))
global_logger.info('GraphLDE ineq. line_l max for test samples: {}'.format(np.mean(test_stats['test_ineq_line_l_max'])))
global_logger.info('\nGraphLDE p_g satisfication rate for test samples: {}'.format(np.mean(test_stats['test_p_g_satisfication rate (%)'])))
global_logger.info('GraphLDE q_g satisfication rate for test samples: {}'.format(np.mean(test_stats['test_q_g_satisfication rate (%)'])))
global_logger.info('GraphLDE v_m satisfication rate for test samples: {}'.format(np.mean(test_stats['test_v_m_satisfication rate (%)'])))
global_logger.info('GraphLDE test_line_limit_satisfication_rate_Sf for test samples: {}'.format(np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)'])))
global_logger.info('GraphLDE test_line_limit_satisfication_rate_St for test samples: {}'.format(np.mean(test_stats['test_line_limit_satisfication_rate_St(%)'])))
global_logger.info('GraphLDE active eq satisfication rate for test samples: {}'.format(np.mean(test_stats['test_active_eq_satisfication rate (%)'])))
global_logger.info('GraphLDE reactive eq satisfication rate for test samples: {}'.format(np.mean(test_stats['test_reactive_eq_satisfication rate (%)'])))
global_logger.info('\nGraphLDE time (ms) <== average value for test dataset: {}'.format((np.mean(solve_time)/1)*1e3))


GraphLDE obj. value for test samples: 12275221.0
GraphLDE eq. mean for test samples: 8.966112545749638e-06
GraphLDE eq. max for test samples: 0.004177061375230551
GraphLDE eq. active mean for test samples: 4.420853201736463e-06
GraphLDE eq. active max for test samples: 0.0007761621382087469
GraphLDE eq. reactive mean for test samples: 1.3511371435015462e-05
GraphLDE eq. reactive max for test samples: 0.0041734520345926285

GraphLDE ineq. mean for test samples: 0.0007925081299617887
GraphLDE ineq. max for test samples: 19.917570114135742
GraphLDE ineq. p_g mean for test samples: 0.09484557807445526
GraphLDE ineq. p_g max for test samples: 19.917570114135742
GraphLDE ineq. q_g mean for test samples: 0.01712515763938427
GraphLDE ineq. q_g max for test samples: 1.1985584497451782
GraphLDE ineq. v_m mean for test samples: 0.0
GraphLDE ineq. v_m max for test samples: 0.0
GraphLDE ineq. line_l mean for test samples: 7.979270577607167e-08
GraphLDE ineq. line_l max for test samples: 0.000870850

GraphLDE obj. value for test samples:  12275221.0
GraphLDE eq. mean for test samples:  8.966113e-06
GraphLDE eq. max for test samples:  0.0041770614
GraphLDE eq. active mean for test samples:  4.420853e-06
GraphLDE eq. active max for test samples:  0.00077616214
GraphLDE eq. reactive mean for test samples:  1.3511371e-05
GraphLDE eq. reactive max for test samples:  0.004173452


GraphLDE ineq. mean for test samples:  0.00079250813
GraphLDE ineq. max for test samples:  19.91757
GraphLDE ineq. p_g mean for test samples:  0.09484558
GraphLDE ineq. p_g max for test samples:  19.91757
GraphLDE ineq. q_g mean for test samples:  0.017125158
GraphLDE ineq. q_g max for test samples:  1.1985584
GraphLDE ineq. v_m mean for test samples:  0.0
GraphLDE ineq. v_m max for test samples:  0.0
GraphLDE ineq. line_l mean for test samples:  7.9792706e-08
GraphLDE ineq. line_l max for test samples:  0.0008708504


GraphLDE p_g satisfication rate for test samples:  99.047615
GraphLDE q_g satisfication rate 

* Harmonic mean

In [9]:
import statistics as st

## Calculate optimality gap
print("GraphLDE obj. value for test samples: ", round(np.mean(test_stats['test_obj_cost'])*10000, 4))
print("GraphLDE eq. mean for test samples: ", st.harmonic_mean(test_stats['test_eq_mean'])) # print("LDF eq. mean for test samples: ", np.mean(test_stats['test_eq_mean']))
print("GraphLDE eq. max for test samples: ", st.harmonic_mean(test_stats['test_eq_max'])) # print("LDF eq. max for test samples: ", np.mean(test_stats['test_eq_max']))
print("GraphLDE eq. active mean for test samples: ", st.harmonic_mean(test_stats['test_eq_real_mean'])) # print("LDF eq. active mean for test samples: ", np.mean(test_stats['test_eq_real_mean']))
print("GraphLDE eq. active max for test samples: ", st.harmonic_mean(test_stats['test_eq_real_max'])) # print("LDF eq. active max for test samples: ", np.mean(test_stats['test_eq_real_max']))
print("GraphLDE eq. reactive mean for test samples: ", st.harmonic_mean(test_stats['test_eq_react_mean'])) # print("LDF eq. reactive mean for test samples: ", np.mean(test_stats['test_eq_react_mean']))
print("GraphLDE eq. reactive max for test samples: ", st.harmonic_mean(test_stats['test_eq_react_max'])) # print("LDF eq. reactive max for test samples: ", np.mean(test_stats['test_eq_react_max']))
print("\n")
print("GraphLDE ineq. mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_mean'])) # print("LDF ineq. mean for test samples: ", np.mean(test_stats['test_ineq_mean']))
print("GraphLDE ineq. max for test samples: ", st.harmonic_mean(test_stats['test_ineq_max'])) # print("LDF ineq. max for test samples: ", np.mean(test_stats['test_ineq_max']))
print("GraphLDE ineq. p_g mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_p_g_mean'])) # print("LDF ineq. p_g mean for test samples: ", np.mean(test_stats['test_ineq_p_g_mean']))
print("GraphLDE ineq. p_g max for test samples: ", st.harmonic_mean(test_stats['test_ineq_p_g_max'])) # print("LDF ineq. p_g max for test samples: ", np.mean(test_stats['test_ineq_p_g_max']))
print("GraphLDE ineq. q_g mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_q_g_mean'])) # print("LDF ineq. q_g mean for test samples: ", np.mean(test_stats['test_ineq_q_g_mean']))
print("GraphLDE ineq. q_g max for test samples: ", st.harmonic_mean(test_stats['test_ineq_q_g_max'])) # print("LDF ineq. q_g max for test samples: ", np.mean(test_stats['test_ineq_q_g_max']))
print("GraphLDE ineq. v_m mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_v_m_mean'])) # print("LDF ineq. v_m mean for test samples: ", np.mean(test_stats['test_ineq_v_m_mean']))
print("GraphLDE ineq. v_m max for test samples: ", st.harmonic_mean(test_stats['test_ineq_v_m_max'])) # print("LDF ineq. v_m max for test samples: ", np.mean(test_stats['test_ineq_v_m_max']))
print("GraphLDE ineq. line_l mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_line_l_mean'])) # print("LDF ineq. line_l mean for test samples: ", np.mean(test_stats['test_ineq_line_l_mean']))
print("GraphLDE ineq. line_l max for test samples: ", st.harmonic_mean(test_stats['test_ineq_line_l_max'])) # print("LDF ineq. line_l max for test samples: ", np.mean(test_stats['test_ineq_line_l_max']))
print("\n")

print("GraphLDE p_g satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_p_g_satisfication rate (%)'].reshape(-1))) # print("LDF p_g satisfication rate for test samples: ", np.mean(test_stats['test_p_g_satisfication rate (%)']))
print("GraphLDE q_g satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_q_g_satisfication rate (%)'].reshape(-1))) # print("LDF q_g satisfication rate for test samples: ", np.mean(test_stats['test_q_g_satisfication rate (%)']))
print("GraphLDE v_m satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_v_m_satisfication rate (%)'].reshape(-1))) # print("LDF v_m satisfication rate for test samples: ", np.mean(test_stats['test_v_m_satisfication rate (%)']))
print("GraphLDE test_line_limit_satisfication_rate_Sf for test samples: ", st.harmonic_mean(test_stats['test_line_limit_satisfication_rate_Sf(%)'].reshape(-1))) # print("LDF test_line_limit_satisfication_rate_Sf for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']))
print("GraphLDE test_line_limit_satisfication_rate_St for test samples: ", st.harmonic_mean(test_stats['test_line_limit_satisfication_rate_St(%)'].reshape(-1))) # print("LDF test_line_limit_satisfication_rate_St for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']))
print("GraphLDE active eq satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_active_eq_satisfication rate (%)'].reshape(-1))) # print("LDF active eq satisfication rate for test samples: ", np.mean(test_stats['test_active_eq_satisfication rate (%)']))
print("GraphLDE reactive eq satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_reactive_eq_satisfication rate (%)'].reshape(-1))) # print("LDF reactive eq satisfication rate for test samples: ", np.mean(test_stats['test_reactive_eq_satisfication rate (%)']))

print("\n")
print("GraphLDE time (ms) <== average value for test dataset:", (np.mean(solve_time)/1)*1e3)


GraphLDE obj. value for test samples:  3459218.4448
GraphLDE eq. mean for test samples:  6.228278545185801e-06
GraphLDE eq. max for test samples:  0.0003987515364172608
GraphLDE eq. active mean for test samples:  2.9959126176184707e-06
GraphLDE eq. active max for test samples:  0.00010875316994758176
GraphLDE eq. reactive mean for test samples:  9.458952113910127e-06
GraphLDE eq. reactive max for test samples:  0.0003986132026868001


GraphLDE ineq. mean for test samples:  6.148506295824064e-07
GraphLDE ineq. max for test samples:  0.010572804678306503
GraphLDE ineq. p_g mean for test samples:  0.0
GraphLDE ineq. p_g max for test samples:  0.0
GraphLDE ineq. q_g mean for test samples:  5.3364120426584116e-05
GraphLDE ineq. q_g max for test samples:  0.010449723644732631
GraphLDE ineq. v_m mean for test samples:  0.0
GraphLDE ineq. v_m max for test samples:  0.0
GraphLDE ineq. line_l mean for test samples:  0.0
GraphLDE ineq. line_l max for test samples:  0.0


GraphLDE p_g satisficatio

/home/super/anaconda3/envs/skjdeep/lib/python3.10/statistics.py:428: RuntimeWarning: divide by zero encountered in divide
  T, total, count = _sum(w / x if w else 0 for w, x in zip(weights, data))


In [10]:
############ UNIFORM DIST. WITH RANDOM SAMPLING ############
# MATPOWER - cost: 3435500, solve time: 5489.90 ms  <=== +/- 20% load perturbation
# GraphLDE - cost: 3459218.4448, solve time: 179.27332890033722 ms <=== +/- 20% load perturbation <=== training time: ~ 38min (37min 51s)

((3459218.4448 - 3435500)/3435500)*100 # <== 20% load perturbation

0.6903928045408287